# ShelfScan — Entrega 1: YOLOv8 Preliminary Training
- **Diego Valenzuela**
- **Daniel Dubon**
- **Bianca Calderon**

In [ ]:
# Install deps (run once)
!pip install ultralytics opencv-python-headless anylabeling kagglehub -q

In [ ]:
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Pre-etiquetar con YOLO-World (zero-shot)

YOLO-World detecta objetos a partir de descripciones de texto sin necesidad de entrenamiento previo.
Genera labels `.txt` en formato YOLO para cada imagen en `data/raw/`.
Después se revisan y corrigen manualmente con labelImg.

In [ ]:
import os, sys
os.chdir('/content/ShelfScan')  # ajustar si es local

sys.path.insert(0, 'scripts')
from autolabel import autolabel_directory

autolabel_directory(
    input_dir='data/raw',
    output_dir='data/annotated',
    conf=0.15,        # umbral bajo — mejor recall para corregir después
)
print("Pre-labels generados en data/annotated/labels/")
print("Revisar y corregir con labelImg antes de entrenar.")

## 2. Run augmentation

In [ ]:
import subprocess, sys, os
os.chdir('/content/ShelfScan')  # adjust if needed
subprocess.run([sys.executable, 'scripts/augmentation.py'], check=True)
subprocess.run([sys.executable, 'scripts/split_dataset.py'], check=True)

## 3. Train YOLOv8n — Preliminary Run

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='data/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    project='models',
    name='shelfscan_v1',
    exist_ok=True,
    plots=True,
)

## 4. Evaluate and report mAP

In [ ]:
best_model = YOLO('models/shelfscan_v1/weights/best.pt')
metrics = best_model.val(data='data/dataset.yaml')

print('=== ShelfScan v1 — Preliminary Metrics ===')
print(f'mAP@0.5:      {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'Precision:    {metrics.box.mp:.4f}')
print(f'Recall:       {metrics.box.mr:.4f}')

# Per-class breakdown
from scripts.categories import CLASS_NAMES
print('\nPer-class mAP@0.5:')
for i, (name, ap) in enumerate(zip(CLASS_NAMES, metrics.box.maps)):
    print(f'  {name:15s}: {ap:.4f}')

## 5. Visualize training curves

In [ ]:
from IPython.display import Image
Image('models/shelfscan_v1/results.png')

## 6. Generar crops

In [ ]:
def generate_crops(results, output_path='data/classifier_dataset'):
    """Toma los resultados de YOLO y guarda cada detección como una imagen individual"""
    if not os.path.exists(output_path):
        os.makedirs(output_path)
    
    for i, res in enumerate(results):
        img = res.orig_img
        for j, box in enumerate(res.boxes):
            # Obtener coordenadas y clase
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls = int(box.cls[0])
            class_name = res.names[cls]
            
            # Crear carpeta por clase
            class_dir = os.path.join(output_path, class_name)
            if not os.path.exists(class_dir):
                os.makedirs(class_dir)
            
            # Recortar y guardar
            crop = img[y1:y2, x1:x2]
            cv2.imwrite(f"{class_dir}/crop_{i}_{j}.jpg", crop)

results = best_model.predict(source='data/test/images', save=False)
generate_crops(results)
print("Recortes generados exitosamente en data/classifier_dataset")

## 7. Transfer Learning

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_data = datasets.ImageFolder('data/classifier_dataset', transform=transform)
train_loader = DataLoader(train_data, batch_size=16, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = models.resnet50(weights='IMAGENET1K_V1')

for param in resnet.parameters():
    param.requires_grad = False

num_classes = len(train_data.classes)
resnet.fc = nn.Linear(resnet.fc.in_features, num_classes)
resnet = resnet.to(device)

print(f"Modelo ResNet-50 listo para entrenar con {num_classes} clases.")

## 8. Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(resnet.fc.parameters(), lr=0.001)

# Entrenamiento de 1 epoca para probar
resnet.train()
for inputs, labels in train_loader:
    inputs, labels = inputs.to(device), labels.to(device)
    optimizer.zero_grad()
    outputs = resnet(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

print("Entrenamiento preliminar completado. Pérdida final:", loss.item())

Propuesta de Analisis Temporal — ShelfScan
Objetivo: Detectar la velocidad de rotación de productos y predecir quiebres de stock.

Recoleccion de datos: Se capturaran imágenes desde una posición fija cada 2 horas y El dataset temporal consistirá en carpetas t1, t2, ..., tn con la estampa de tiempo.

Deteccion de Cambios: Se utilizará la comparación de coordenadas de YOLO entre t_n y t_{n-1}. Si un objeto desaparece y se detecta una zona vacía en su lugar, se marca como "unidad vendida".

Métricas de Prediccion:

V_vta: Velocidad de venta (unidades/hora).

T_out: Tiempo estimado para agotamiento = Unidades_Actuales / V_vta.

Algoritmo: Se implementará una regresión lineal simple sobre el conteo de productos por categoría para estimar el momento exacto en que el stock llegará a cero.